##**[11주차]실습**
- 교재를 참고하여 아래의 실습을 수행하시오.
- 모든 코드의 결과를 출력하여, .ipynb의 링크를 **[11주차]/[11주차]과제**에 제출하시오.\
(실습 제출 예시: 11주차_2020XXXX_이름.ipynb 코드 링크)

In [1]:
print("2343035 신동엽")

2343035 신동엽


In [18]:
# 설치 후 세션 재시작, 재시작 후에는 해당 셀 스킵
!pip install numpy regex==2017.4.5 requests==2.27.1 tqdm==4.64.0 fire==0.5.0

  Using cached regex-2017.4.5-cp312-cp312-linux_x86_64.whl
  Attempting uninstall: regex
    Found existing installation: regex 2026.5.9
    Uninstalling regex-2026.5.9:
      Successfully uninstalled regex-2026.5.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nltk 3.9.1 requires regex>=2021.8.3, but you have regex 2017.4.5 which is incompatible.
tiktoken 0.12.0 requires regex>=2022.1.18, but you have regex 2017.4.5 which is incompatible.


In [2]:
from google.colab import auth

auth.authenticate_user()
!gcloud config get-value account

/usr/local/lib/python3.12/dist-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


2343035@donga.ac.kr


In [3]:
# google drive 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import sys

# Add the directory containing utils.py to sys.path
sys.path.append('/content/drive/MyDrive/deepLearning/11주차')

In [5]:
!pip install --upgrade regex

  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (801 kB)
  Attempting uninstall: regex
    Found existing installation: regex 2017.4.5
    Uninstalling regex-2017.4.5:
      Successfully uninstalled regex-2017.4.5


###**예제 1) picoGPT의 decoder를 작성해본다.**
- **[실습목표]** picoGPT의 decode를 작성하며, transformer의 구조를 파악해본다.


In [6]:
import numpy as np
from tqdm import tqdm
from utils import load_encoder_hparams_and_params

#### **1) main() 메소드**
- GPT-2모델을 사용하여 텍스트 생성을 실행할 때 필요한 입력값을 설정
- 아래의 각 매개변수의 역할을 서술하시오.
  - prompt(str): gpt 모델이 텍스트 생성을 시작하기 위해 참조하는 입력 문자열
  - n_tokens_to_generate(int): gpt 신경망의 최종 연산 결과로 도출되는 비정규화된 확률 점수 행렬

In [7]:
def main(prompt: str, n_tokens_to_generate: int = 40, model_size: str = "124M", models_dir: str = "models"):
    encoder, hparams, params = load_encoder_hparams_and_params(model_size, models_dir)
    input_ids = encoder.encode(prompt)
    assert len(input_ids) + n_tokens_to_generate < hparams["n_ctx"]
    output_ids =  generate(input_ids, params, hparams["n_head"], n_tokens_to_generate)
    output_text = encoder.decode(output_ids)
    return output_text

#### **2) generate() 메소드**
- 주어진 입력 시퀀스에 대해 추가적인 토큰을 생성하는 함수
- generate()함수 내에서 gpt2 함수를 통해 입력 시퀀스에 대한 예측을 수행
- 아래의 각 변수의 역할에 대해 서술
  - logits: GPT-2 신경망의 최종 연산 결과로 도출되는 비정규화된 확률 점수 행렬


In [8]:
def generate(inputs, params, n_head, n_tokens_to_generate):
    for _ in tqdm(range(n_tokens_to_generate), "generating"):
        logits = gpt2(inputs, **params, n_head=n_head)
        next_id = np.argmax(logits[-1])
        inputs.append(int(next_id))
    return inputs[len(inputs) - n_tokens_to_generate :]

#### **3) gpt2() 메소드**
- 입력 시퀀스에 대해 예측을 수행

In [9]:
def gpt2(inputs, wte, wpe, blocks, ln_f, n_head):
    x = wte[inputs] + wpe[range(len(inputs))]
    for block in blocks:
        x = transformer_block(x, **block, n_head=n_head)
    x = layer_norm(x, **ln_f)
    return x @ wte.T

#### **4) transformer_block()메소드**
- 멀티헤드 어텐션(Multi Head Attention, MHA)와 피드-포워드 신경망(Feed Foward Network)를 포함하는 트렌스포머 블록을 구현한 함수



In [10]:
def transformer_block(x, mlp, attn, ln_1, ln_2, n_head):
    x = x + mha(layer_norm(x, **ln_1), **attn, n_head=n_head)
    x = x + x + ffn(layer_norm(x, **ln_2), **mlp)
    return x

#### **5) mha()**
- 멀티헤드 어텐션함수는 입력 벡터를 여러 개의 어텐션 헤드로 나누어 병렬로 어텐션을 계산한 후, 그 결과를 결합하는 역할
- 입력 벡터를 여러개의 어텐션 헤드로 나누는 이유는 무엇인가?
- : 데이터를 병렬적으로 처리하기 위해서


In [11]:
def mha(x, c_attn, c_proj, n_head):
    x = linear(x, **c_attn)
    qkv_heads = list(map(lambda x: np.split(x, n_head, axis=-1), np.split(x, 3, axis=-1)))
    causal_mask = (1 - np.tri(x.shape[0], dtype=x.dtype)) * -1e10
    out_heads = [attention(q, k, v, causal_mask) for q, k, v in zip(*qkv_heads)]
    x = linear(np.hstack(out_heads), **c_proj)
    return x

#### **6) attention() 메소드**
- 어텐션 함수는 주어진 쿼리(Q), 키(K), 값(V) 및 마스크를 사용하여 어텐션 가중치를 계산하는 함수이다.


In [12]:
def attention(q, k, v, mask):
    return softmax(q @ k.T / np.sqrt(q.shape[-1])) @ v

#### **7) ffn() 메소드**
- 피드-포워드 신경망 함수는 트렌스포머 모델에서 사용되는 구성 요소 중 하나로, 입력 벡터에 대해 두 번의 선형 변환과 활성화 함수를 적용하여 출력 벡터를 생성하는 함수이다.
- 멀티헤드 어텐션 이후에 적용.

In [13]:
def ffn(x, c_fc, c_proj):
    return linear(gelu(linear(x, **c_fc)), **c_proj)

#### **8, 9, 10) linear(), layer_norm(), gelu(), softmax() 메소드**
- 각각 선형 변환 함수, 레이어 정규화 함수, GELU 활성화 함수, Softmax 함수를 의미하며, 각 함수의 역할은 아래의 설명과 같다.
- **linear()**: 입력 벡터에 대해 선형 변환을 수행.
- **layer_norm()**: 각 레이어의 출력에 정규화를 수행하여 학습 안정성을 높이고, 수렴 속도를 향상.
- **gelu()**: 입력 값을 비선형적으로 변환하여 모델의 표현력을 향상
- **softmax()**: 입력 벡터를 확률 분포로 변환하는 함수, 출력 벡터의 요소들을 0과 1 사이의 값으로 매핑 시키며, 전체 합이 1이 되도록 한다.

In [14]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, g, b, eps: float = 1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return g * (x - mean) / np.sqrt(variance + eps) + b

def linear(x, w, b):
    return x @ w + b

#### **11) prompt 입력 후 결과 출력**


In [15]:
if __name__ == "__main__":
    print(main(prompt="Alan Turing theorized that computers would one day become"))
    # prompt를 여러가지로 변경하며, 테스트 해보기(선택사항)

generating: 100%|██████████| 40/40 [00:56<00:00,  1.41s/it]

 a new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new new
